In [ ]:
"""from datasets import load_dataset

# Loading 50,000 stories
dataset = load_dataset("roneneldan/TinyStories", split="train[:50000]")

# Saving everything as a text file
with open("data/tinystories_extract.txt", "w", encoding="utf-8") as f:
    for story in dataset:
        f.write(story['text'] + "\n<|endoftext|>\n")"""

# Only needs to be ran once

In [2]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel

tokenizer = Tokenizer(BPE())
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)

vocab_size = 1000

trainer = BpeTrainer(
    vocab_size=vocab_size,
    special_tokens=["<|endoftext|>"],
    show_progress=True
)

file = ["data/tinystories_extract.txt"]

tokenizer.train(file, trainer=trainer)

tokenizer.save(f"weights/tiny_stories_50ktrain_{vocab_size}.json")
print("BPE training completed. Vocab size is :", tokenizer.get_vocab_size())
# Only needs to be ran once too




BPE training completed. Vocab size is : 1000


In [3]:
import numpy as np
from src.ai_lib import layers
from src.ai_lib import Model, metrics, SequenceDataLoader, CosineWarmupScheduler
from src.ai_lib import optimizers
from src.ai_lib import losses
from src.ai_lib import preprocessing

vocab_size = 1000

tokenizer = preprocessing.CustomBPETokenizer(path= f"weights/tiny_stories_50ktrain_{vocab_size}.json")

with open("data/tinystories_extract.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

import numpy as np
data_tensor = np.array(tokenizer.encode(text_data), dtype=np.int32)

print(tokenizer.vocab_size)

1000


In [4]:
train_test_split = 0.95

n = int(train_test_split * len(data_tensor))
train_data = data_tensor[:n]
val_data = data_tensor[n:]

batch_size = 64
block_size = 64  # Numbers of tokens to look back at
steps_per_epoch = 25 

# Creating dataloaders
train_loader = SequenceDataLoader(
    data=train_data, 
    block_size=block_size, 
    batch_size=batch_size, 
    steps_per_epoch=steps_per_epoch
)

val_loader = SequenceDataLoader(
    data=val_data, 
    block_size=block_size, 
    batch_size=batch_size, 
    steps_per_epoch=5 # Less steps for validation
)

In [6]:
from functools import partial
d_model = 128
n_heads = 8
d_ff = int(128 * 8/3)
n_kv_heads = 4

dropout_rate = 0.1

vocab_size = tokenizer.vocab_size

norm_builder = partial(layers.RMSNorm, n_features=d_model)

attn_builder = partial(layers.GQARoPEAttentionHead, n_heads=n_heads, n_kv_heads = n_kv_heads, d_model=d_model, is_causal=True, block_size=block_size)

expert_builder = partial(layers.SwiGLU, in_features=d_model, hidden_features=d_ff)

ffn_builder = partial(layers.MoEBlock, expert_builder,d_model)

embedding_layer = layers.Embedding(vocab_size=vocab_size, d_model=d_model)

llm_network = layers.Sequential([
    embedding_layer,

    # 6 Transformers
    layers.TransformerBlock(attention_builder=attn_builder, norm_builder=norm_builder,ffn_builder=ffn_builder, dropout_rate=dropout_rate),
    layers.TransformerBlock(attention_builder=attn_builder, norm_builder=norm_builder,ffn_builder=ffn_builder, dropout_rate=dropout_rate),
    layers.TransformerBlock(attention_builder=attn_builder, norm_builder=norm_builder,ffn_builder=ffn_builder, dropout_rate=dropout_rate),
    layers.TransformerBlock(attention_builder=attn_builder, norm_builder=norm_builder,ffn_builder=ffn_builder, dropout_rate=dropout_rate),
    layers.TransformerBlock(attention_builder=attn_builder, norm_builder=norm_builder,ffn_builder=ffn_builder, dropout_rate=dropout_rate),
    layers.TransformerBlock(attention_builder=attn_builder, norm_builder=norm_builder,ffn_builder=ffn_builder, dropout_rate=dropout_rate),

    layers.RMSNorm(n_features=d_model),

    layers.TiedOutputProjection(embedding_layer)
])

model = Model(llm_network)

In [10]:
learning_rate = 1e-3
weight_decay = 1e-1

optimizer = optimizers.Adam(learning_rate=learning_rate, weight_decay=weight_decay)
loss = losses.SoftmaxCrossEntropy()

lr_scheduler = CosineWarmupScheduler(lr_max = 5e-3, lr_min = 5e-4, warmup_steps=100, total_steps=32000)

In [19]:
print("Beginning of training of microgpt_v3")
model.fit(
    dataloader=train_loader,
    epochs=30,
    loss=loss,
    optimizer=optimizer,
    validation_dataloader=val_loader,
    verbose=True,
    lr_scheduler=lr_scheduler,
    metrics=[metrics.accuracy]
)


Beginning of training of microgpt_v3

--- Epoch : 0 ---
Training loss : 2.36929
  accuracy (train) : 0.45666
Validation loss : 2.28116
  accuracy (val) : 0.46802

--- Epoch : 1 ---
Training loss : 2.30936
  accuracy (train) : 0.46675
Validation loss : 2.25918
  accuracy (val) : 0.47998

--- Epoch : 2 ---
Training loss : 2.31549
  accuracy (train) : 0.46349
Validation loss : 2.29079
  accuracy (val) : 0.47348

--- Epoch : 3 ---
Training loss : 2.33590
  accuracy (train) : 0.46171
Validation loss : 2.31355
  accuracy (val) : 0.46206

--- Epoch : 4 ---
Training loss : 2.35923
  accuracy (train) : 0.45572
Validation loss : 2.34410
  accuracy (val) : 0.45933

--- Epoch : 5 ---
Training loss : 2.38761
  accuracy (train) : 0.45056
Validation loss : 2.32101
  accuracy (val) : 0.45864

--- Epoch : 6 ---
Training loss : 2.36994
  accuracy (train) : 0.45311
Validation loss : 2.32505
  accuracy (val) : 0.46699

--- Epoch : 7 ---
Training loss : 2.37522
  accuracy (train) : 0.45361
Validation loss 

KeyboardInterrupt: 

In [18]:
def generate_text(model, tokenizer, prompt_text="A ", max_new_tokens=100):
    """
    Generates text
    """
    # Important : Turns off setting
    model.sequential.set_training(False)
    
    # Use of kv_cache for inference
    model.sequential.set_use_cache(True)
    model.sequential.reset_cache()
    
    # Encodes context (initial text / prompt)
    context_ids = tokenizer.encode(prompt_text)
    
    # Prefill
    X_input = np.array([context_ids], dtype=np.int32)
    logits = model.predict(X_input)
    
    # We take the predicted char and add it to the list
    next_char_id = sample_top_k(logits[0, -1, :], temperature=1, top_k=3)
    context_ids.append(next_char_id)
    
    # Decode
    for _ in range(max_new_tokens - 1):
        # X_input becomes a simple integer as we use kv_cache
        X_input = np.array([[next_char_id]], dtype=np.int32)
        
        logits = model.predict(X_input)
        
        # Since X_input is of length one, logits has shape (1, 1, Vocab_size)
        next_char_id = sample_top_k(logits[0, -1, :], temperature=1, top_k=3)
        context_ids.append(next_char_id)
        
    return tokenizer.decode(context_ids)


def sample_top_k(logits, temperature=1, top_k=5):
    """
    Sampling from only the k most likely tokens with temperature
    """

    logits = logits / temperature
    
    indices_to_remove = np.argsort(logits)[:-top_k]
    
    logits[indices_to_remove] = -np.inf
    
    exp_preds = np.exp(logits - np.max(logits))
    preds = exp_preds / np.sum(exp_preds)
    
    return np.random.choice(len(logits), p=preds)

generated_text = generate_text(model, tokenizer, prompt_text="Once upon a time", max_new_tokens=300)
print(generated_text)

Once upon a time, there was a little girl named Lily. She loved to play outside in the park. One day, Lily's mom gave her a big bag of paper and a paper. Lily was so excited to help her mommy.

But then, she heard a noise coming from behind her. She looked up and saw the bucket was very high. She saw the pains and wanted to play.

Lily went to the store with her mommy. She saw that shells and wanted to play with her mommy. Lily washing she saw a big, sheacht, Lily was a little bit a big, Lily's moment. She asked her mom, "Can I do, Lily?" Herativailable, Lily went to go to goat the sleep. She was playing, Lily's mom called her to go for a walk in the park. Lily wanted to play with her dolls and make things. She wanted to play with her toy cars and make things better. She asked Lily to help her mom clean up her toys, but she knew she had to share it with her. Her mom said yes, but Lily didn't listen.

Later that day, Lily went to her room and saw the toy car. She wanted to play with her

In [ ]:
model.save_weights('weights/moe.npz')